# Distributive Computing — Assignment 2
## Part 1: Data Pipeline (PySpark)

**Research Question:** Can FinBERT sentiment analysis of financial news articles predict extreme stock price movements (>1.5% in one day)?

### Pipeline Overview
| Cell | Stage | Skip if... |
|------|-------|------------|
| 1 | Setup — imports, paths, checkpoint utilities | Never |
| 2 | Download raw dataset HuggingFace → GCS | File already in GCS |
| 3 | Verify GCS bucket contents | — |
| 4 | Preview raw CSV (first 5 rows) | — |
| 5 | Load raw CSV into Spark | checkpoint 01 exists |
| 6 | Clean & preprocess | checkpoint 01 exists |
| 7 | Inspect columns after cleaning | checkpoint 01 exists |
| 8 | Explore stock coverage & justify selection | checkpoint 01 exists |
| 9 | Verify dataset shape | checkpoint 01 exists |
| 10 | Drop unused columns & cache | checkpoint 01 exists |
| 11 | Filter to 15 target stocks | checkpoint 02 exists |
| 12 | Save filtered dataset to GCS (Parquet) | checkpoint 02 exists |
| 13 | Download historical prices (yfinance) | checkpoint 03 exists |
| 14 | Verify date ranges & preview prices | checkpoint 03 exists |
| 15 | Reshape prices to long format | checkpoint 03 exists |
| 16 | Join news with prices (Spark) | checkpoint 03 exists |
| 17 | Inspect joined dataset | — |
| 18 | Convert to pandas | checkpoint 04 exists |
| 19 | VADER sentiment (baseline comparison) | checkpoint 04 exists |
| 20 | Inspect VADER results manually | checkpoint 04 exists |
| 21 | FinBERT sentiment (final model) | checkpoint 04 exists |
| 22 | Save final checkpoint to GCS | — |

**Dataset:** FNSPID — Financial News and Stock Price Integration Dataset  
**Source:** https://huggingface.co/datasets/Zihan1004/FNSPID  
**Size:** ~2.4M articles with pre-computed extractive summaries (LSA, Luhn, TextRank, LexRank)

In [ ]:
# ============================================================
# [CELL 1] SETUP — IMPORTS, PATHS, CHECKPOINT UTILITIES
# ============================================================
# All intermediate results are saved to GCS so they persist
# across Dataproc cluster sessions. If the cluster restarts,
# we load from the last checkpoint instead of recomputing.
#
# ALWAYS RUN THIS CELL — defines all variables and functions
# used by every subsequent cell.

import os
import subprocess
import pickle
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.functions import to_date, min, max

# ── Global paths ─────────────────────────────────────────────
GCS_BUCKET     = 'gs://assesment2-dc'
NEWS_FILE      = 'fnspid_news.csv'
NEWS_GCS_PATH  = f'{GCS_BUCKET}/{NEWS_FILE}'
NEWS_URL       = (
    'https://huggingface.co/datasets/Zihan1004/FNSPID'
    '/resolve/main/Stock_news/nasdaq_exteral_data.csv'
)
PARQUET_PATH   = f'{GCS_BUCKET}/fnspid_filtered/'
CHECKPOINT_DIR = f'{GCS_BUCKET}/checkpoints'

# ── Target stocks ─────────────────────────────────────────────
# 15 liquid, well-covered NASDAQ stocks selected based on:
#   1. News coverage >= 8,000 articles in FNSPID
#   2. Historical price data available via yfinance (2008-2024)
#   3. Sector diversity for generalizability
#
# Sectors:
#   Tech:          AAPL, GOOG, MSFT, NVDA, AMD, TSLA, INTC
#   Finance:       GS, WFC
#   Retail/Media:  WMT, DIS
#   Energy:        CVX
#   Telecom:       T
#   Industrial:    GE
#   International: BABA
target_stocks = [
    'AAPL', 'GOOG', 'MSFT', 'NVDA', 'AMD', 'TSLA',
    'BABA', 'GE',   'DIS',  'WMT',
    'GS',   'WFC',  'CVX',  'T',   'INTC'
]

# ── Checkpoint utilities ──────────────────────────────────────

def file_exists_in_gcs(path):
    """
    Check if a file or directory exists in GCS.
    Return code 0 = exists, non-zero = not found.
    """
    result = subprocess.run(
        ['gcloud', 'storage', 'ls', path],
        capture_output=True
    )
    return result.returncode == 0


def save_checkpoint(name, spark_df=None, pandas_df=None, metadata=None):
    """
    Save intermediate results to GCS.
      spark_df   -> Parquet (columnar, compressed, splittable by Spark)
      pandas_df  -> CSV
      metadata   -> pickle (Python dict)
    """
    path = f'{CHECKPOINT_DIR}/{name}'
    if spark_df is not None:
        spark_df.write.mode('overwrite').parquet(f'{path}_spark/')
        print(f'  [SAVED] Spark DF    -> {path}_spark/')
    if pandas_df is not None:
        tmp = f'/tmp/{name}.csv'
        pandas_df.to_csv(tmp, index=False)
        os.system(f'gcloud storage cp {tmp} {path}.csv')
        print(f'  [SAVED] Pandas DF   -> {path}.csv')
    if metadata is not None:
        tmp = f'/tmp/{name}_meta.pkl'
        with open(tmp, 'wb') as f:
            pickle.dump(metadata, f)
        os.system(f'gcloud storage cp {tmp} {path}_meta.pkl')
        print(f'  [SAVED] Metadata    -> {path}_meta.pkl')


def load_checkpoint(name, spark=None, load_spark=False,
                    load_pandas=False, load_meta=False):
    """
    Load a previously saved checkpoint from GCS.
    Returns dict with keys: spark_df, pandas_df, metadata.
    """
    path    = f'{CHECKPOINT_DIR}/{name}'
    results = {}
    if load_spark and spark is not None:
        df = spark.read.parquet(f'{path}_spark/')
        df.cache()
        print(f'  [LOADED] Spark DF   <- {path}_spark/ ({df.count():,} rows)')
        results['spark_df'] = df
    if load_pandas:
        tmp = f'/tmp/{name}.csv'
        os.system(f'gcloud storage cp {path}.csv {tmp}')
        df = pd.read_csv(tmp)
        print(f'  [LOADED] Pandas DF  <- {path}.csv ({len(df):,} rows)')
        results['pandas_df'] = df
    if load_meta:
        tmp = f'/tmp/{name}_meta.pkl'
        os.system(f'gcloud storage cp {path}_meta.pkl {tmp}')
        with open(tmp, 'rb') as f:
            meta = pickle.load(f)
        print(f'  [LOADED] Metadata:  {meta}')
        results['metadata'] = meta
    return results


def checkpoint_exists(name, suffix='_spark'):
    """Check if a checkpoint already exists in GCS."""
    path   = f'{CHECKPOINT_DIR}/{name}{suffix}/'
    result = subprocess.run(
        ['gcloud', 'storage', 'ls', path],
        capture_output=True
    )
    return result.returncode == 0


# ── Check what checkpoints already exist ─────────────────────
# This tells you exactly which cells you can skip on this run.
print('=' * 55)
print('CHECKPOINT STATUS')
print('=' * 55)
checks = {
    '01_df_clean  (cleaned dataset)':      checkpoint_exists('01_df_clean'),
    '02_df_stocks (filtered 15 stocks)':   checkpoint_exists('02_df_stocks'),
    '03_df_joined (joined with prices)':   checkpoint_exists('03_df_joined'),
    '04_pd_news   (FinBERT labeled)':      file_exists_in_gcs(f'{CHECKPOINT_DIR}/04_pd_news.csv'),
}
for name, exists in checks.items():
    status = 'EXISTS  -> SKIP cells that produce this' if exists else 'MISSING -> run cells to produce this'
    print(f'  {name}: {status}')

print('\nSetup complete.')
print(f'  GCS Bucket:      {GCS_BUCKET}')
print(f'  Checkpoint dir:  {CHECKPOINT_DIR}')
print(f'  Target stocks:   {target_stocks}')

In [ ]:
# ============================================================
# [CELL 2] DOWNLOAD RAW DATASET FROM HUGGINGFACE -> GCS
# ============================================================
# SKIP IF: file already in GCS (checked automatically below)
#
# The raw FNSPID dataset (~2.4M rows, ~3GB) is downloaded
# once directly to GCS using a streaming pipe:
#   curl (download) -> gcloud storage cp (write to GCS)
#
# Streaming avoids saving the file locally first, which would
# require ~3GB of local disk on the cluster node.

if not file_exists_in_gcs(NEWS_GCS_PATH):
    print('File not found in GCS. Downloading FNSPID dataset...')
    print(f'  Source:      {NEWS_URL}')
    print(f'  Destination: {NEWS_GCS_PATH}')
    # Stream directly from HuggingFace to GCS:
    # curl -L follows redirects, pipes stdout to
    # gcloud storage cp which reads from stdin (-)
    os.system(f'curl -L "{NEWS_URL}" | gcloud storage cp - {NEWS_GCS_PATH}')
    print('Download complete!')
else:
    print(f'[SKIP] Dataset already in GCS at {NEWS_GCS_PATH}')
    print('Skipping download.')

In [ ]:
# ============================================================
# [CELL 3] VERIFY GCS BUCKET CONTENTS
# ============================================================
# List all files in the bucket with sizes and timestamps.
# Confirms the file uploaded correctly and is not empty.
# gsutil ls -l shows: file size (bytes), timestamp, path.

result = subprocess.run(
    ['gsutil', 'ls', '-l', f'{GCS_BUCKET}/'],
    capture_output=True, text=True
)
print('GCS Bucket contents:')
print(result.stdout)
if result.returncode != 0:
    print('ERROR accessing bucket:')
    print(result.stderr)

In [ ]:
# ============================================================
# [CELL 4] PREVIEW RAW CSV (first 5 rows)
# ============================================================
# Stream the first 5 lines directly from GCS without
# downloading the full 3GB file. Confirms:
#   1. The file is readable from GCS
#   2. The schema looks correct (column names, separators)
#   3. The data is not corrupted
#
# gsutil cat streams file contents to stdout.
# head -n 5 stops after 5 lines.
# shell=True is required to use the pipe (|) operator.

result = subprocess.run(
    f'gsutil cat {NEWS_GCS_PATH} | head -n 5',
    shell=True, capture_output=True, text=True
)
print('First 5 rows of raw CSV (streamed from GCS):')
print(result.stdout)
if result.returncode != 0:
    print('ERROR reading file:')
    print(result.stderr)

In [ ]:
# ============================================================
# [CELL 5] LOAD RAW CSV INTO SPARK
# ============================================================
# SKIP IF: checkpoint 01_df_clean already exists in GCS
#
# Read the full FNSPID CSV (~2.4M rows) into a Spark DataFrame.
# Spark automatically partitions the file across all cluster
# workers for parallel processing.
#
# CSV parsing options:
#   header=true         -> first row contains column names
#   quote/escape='"'    -> handle double-quoted fields (RFC 4180)
#   multiLine=true      -> article texts may span multiple lines
#   mode=DROPMALFORMED  -> skip unparseable rows (tiny fraction)
#   maxCharsPerColumn   -> raised for very long article texts

if checkpoint_exists('01_df_clean'):
    print('[SKIP] Checkpoint 01_df_clean exists — loading from GCS...')
    ck       = load_checkpoint('01_df_clean', spark=spark, load_spark=True)
    df_clean = ck['spark_df']
    row_count = df_clean.count()
    print(f'Loaded: {row_count:,} rows | {len(df_clean.columns)} columns')
else:
    print('Loading raw CSV into Spark...')
    df = spark.read \
        .option('header',            'true') \
        .option('quote',             '"') \
        .option('escape',            '"') \
        .option('multiLine',         'true') \
        .option('mode',              'DROPMALFORMED') \
        .option('maxCharsPerColumn', '999999') \
        .csv(NEWS_GCS_PATH)

    print('Raw dataset loaded into Spark.')
    print('\nSchema:')
    df.printSchema()
    print(f'Total columns: {len(df.columns)}')
    print(f'Column names:  {df.columns}')

In [ ]:
# ============================================================
# [CELL 6] CLEAN & PREPROCESS DATASET
# ============================================================
# SKIP IF: checkpoint 01_df_clean already exists in GCS
#
# All Spark operations are LAZY — nothing executes until an
# action (.count(), .show(), .write()) is called at the end.
#
# Steps:
#   1. Drop auto-generated index column (Unnamed: 0)
#   2. Parse date string -> TimestampType
#   3. Remove rows with NULL in critical columns
#   4. Trim whitespace from string columns
#   5. Deduplicate articles

if checkpoint_exists('01_df_clean'):
    print('[SKIP] Checkpoint 01_df_clean exists — cleaning already done.')
    print('       df_clean is already loaded from Cell 5.')
else:
    # Step 1 — Drop auto-generated index column
    # The CSV was exported from pandas and contains 'Unnamed: 0',
    # a sequential row number with no analytical value.
    df = df.drop('Unnamed: 0')

    # Step 2 — Parse date string -> TimestampType
    # Raw Date: '2022-06-03 00:00:00 UTC' (StringType)
    # Target: TimestampType — required for filtering, sorting,
    # and joining with price data in [CELL 16].
    df = df.withColumn(
        'Date',
        F.to_timestamp('Date', 'yyyy-MM-dd HH:mm:ss z')
    )

    # Step 3 — Remove rows with NULL in critical columns
    # Keep rows only when ALL FOUR fields are present:
    #   Date          -> join key with price data
    #   Article_title -> deduplication key
    #   Stock_symbol  -> group/filter key
    #   Article       -> source of pre-computed summaries
    df_clean = df.filter(
        F.col('Date').isNotNull() &
        F.col('Article_title').isNotNull() &
        F.col('Stock_symbol').isNotNull() &
        F.col('Article').isNotNull()
    )

    # Step 4 — Trim whitespace from string columns
    # 'AAPL ' != 'AAPL' in Spark -> phantom stock symbol,
    # breaks all downstream joins and aggregations.
    for col_name in ['Article_title', 'Publisher', 'Author', 'Stock_symbol']:
        df_clean = df_clean.withColumn(col_name, F.trim(F.col(col_name)))

    # Step 5 — Deduplicate articles
    # Some articles are syndicated across multiple publishers.
    # Key: (Date, Article_title, Stock_symbol).
    # Without this, the same event inflates daily sentiment scores.
    df_clean = df_clean.dropDuplicates(['Date', 'Article_title', 'Stock_symbol'])

    # Trigger execution
    row_count = df_clean.count()
    print(f'Cleaned dataset: {row_count:,} rows | {len(df_clean.columns)} columns')
    print('\nSample rows after cleaning:')
    df_clean.show(3, truncate=80)

    # Save checkpoint
    save_checkpoint('01_df_clean', spark_df=df_clean)

In [ ]:
# ============================================================
# [CELL 7] INSPECT COLUMNS AFTER CLEANING
# ============================================================
# Verify the columns remaining after the cleaning step.
#
# The 4 summary columns are pre-computed extractive summaries:
#   Lsa_summary      -> Latent Semantic Analysis
#   Luhn_summary     -> Luhn keyword-frequency algorithm
#   Textrank_summary -> Graph-based ranking (PageRank for text)
#   Lexrank_summary  -> Cosine similarity graph ranking
#
# WHY SUMMARIES AND NOT FULL ARTICLES FOR FINBERT?
#   FinBERT (BERT-based) has a hard 512-token input limit.
#   Full articles are often 5,000+ tokens. LSA summaries
#   capture the most semantically relevant sentences and fit
#   within the 512-token limit.

print(f'Dataset shape: ({df_clean.count():,} rows, {len(df_clean.columns)} columns)')
print('\nColumns remaining after cleaning:')
for i, col in enumerate(df_clean.columns):
    print(f'  {i+1:2d}. {col}')

print('\nSample Lsa_summary values (input to FinBERT):')
df_clean.select('Stock_symbol', 'Date', 'Lsa_summary') \
    .filter(F.col('Lsa_summary').isNotNull()) \
    .show(3, truncate=120)

In [ ]:
# ============================================================
# [CELL 8] EXPLORE STOCK COVERAGE -> JUSTIFY SELECTION
# ============================================================
# Count articles per stock to justify our choice of 15 stocks.
#
# SELECTION CRITERIA:
#   1. News coverage >= 8,000 articles in FNSPID
#      -> fewer articles = unreliable daily sentiment averages
#      -> sparse coverage = many trading days with no signal
#   2. Price data availability via yfinance (2008-2024)
#   3. Sector diversity for result generalizability
#
# WHY THESE SPECIFIC STOCKS?
#   Tech stocks (AAPL, GOOG, NVDA, AMD) are heavily news-driven:
#   earnings surprises, product launches, and regulatory news
#   cause immediate price reactions — ideal for testing whether
#   sentiment predicts price moves.
#
#   Finance (GS, WFC), Energy (CVX), and Telecom (T) provide
#   contrast — more stable, less news-sensitive — allowing
#   comparison of sentiment signal strength by sector.
#
#   BABA adds international exposure (China tech regulatory risk).

print('Article count per stock (full dataset, top 50):')
print('Stocks with 8,000+ articles are candidates for selection.\n')

df_clean.groupBy('Stock_symbol') \
    .count() \
    .orderBy('count', ascending=False) \
    .show(50)

print('\nDate range of full dataset:')
df_clean.select(min('Date'), max('Date')).show()

print('\nSelected stocks and justification:')
selection = [
    ('AAPL', 'Tech',          'Apple — highest coverage, most news-driven'),
    ('GOOG', 'Tech',          'Alphabet — major coverage, ad/AI news'),
    ('MSFT', 'Tech',          'Microsoft — cloud/AI, consistent coverage'),
    ('NVDA', 'Tech',          'NVIDIA — AI chip boom, high volatility'),
    ('AMD',  'Tech',          'AMD — volatile, chip sector competitor'),
    ('TSLA', 'Tech',          'Tesla — high retail attention, volatile'),
    ('INTC', 'Tech',          'Intel — mature chip maker, sector contrast'),
    ('BABA', 'International', 'Alibaba — China tech, regulatory risk news'),
    ('GE',   'Industrial',    'GE — broad conglomerate, stable news flow'),
    ('DIS',  'Retail/Media',  'Disney — entertainment, content-driven'),
    ('WMT',  'Retail',        'Walmart — defensive, earnings-focused'),
    ('GS',   'Finance',       'Goldman Sachs — market sentiment indicator'),
    ('WFC',  'Finance',       'Wells Fargo — banking sector'),
    ('CVX',  'Energy',        'Chevron — oil price sensitive news'),
    ('T',    'Telecom',       'AT&T — stable, dividend stock, low volatility'),
]
print(f'  {"Ticker":<6} {"Sector":<15} Reason')
print(f'  {"-"*6} {"-"*15} {"-"*40}')
for ticker, sector, reason in selection:
    print(f'  {ticker:<6} {sector:<15} {reason}')

In [ ]:
# ============================================================
# [CELL 9] VERIFY DATASET SHAPE
# ============================================================
# Print total rows and columns — Spark equivalent of df.shape.
# Expected: ~2,467,690 rows, 9 columns after cleaning.
# Confirms we retained the vast majority of the 2.4M articles
# (raw data was already relatively clean).
#
# Note: .count() triggers a full Spark job (~1-2 min).

row_count = df_clean.count()
col_count = len(df_clean.columns)

print(f'Dataset shape: ({row_count:,} rows, {col_count} columns)')
print(f'  Rows:    {row_count:,}')
print(f'  Columns: {col_count} -> {df_clean.columns}')

In [ ]:
# ============================================================
# [CELL 10] DROP UNUSED COLUMNS & CACHE
# ============================================================
# SKIP IF: checkpoint 01_df_clean already exists
#
# Drop the full article text and unused metadata columns.
# We keep only the pre-computed summaries for FinBERT input.
#
# WHY DROP ARTICLE, AUTHOR, URL?
#   Article -> full text, often thousands of words.
#             FinBERT has a 512-token limit so we use
#             pre-computed summaries. Dropping saves
#             ~3x memory across all cluster workers.
#   Author  -> not relevant for price prediction.
#   Url     -> not relevant for price prediction.
#
# WHY CACHE?
#   Without cache, every subsequent Spark action re-reads
#   the full 3GB CSV from GCS from scratch (slow).
#   With cache, all transformations are computed ONCE and
#   stored in cluster memory across all workers.

if checkpoint_exists('01_df_clean'):
    print('[SKIP] Checkpoint 01_df_clean exists — columns already dropped and cached.')
    print(f'       Current columns: {df_clean.columns}')
else:
    df_clean = df_clean.drop('Article', 'Author', 'Url')
    df_clean.cache()
    row_count = df_clean.count()
    print(f'Cached dataset: {row_count:,} rows | {len(df_clean.columns)} columns')
    print(f'Remaining columns: {df_clean.columns}')
    print('\nDataset is now cached in cluster memory.')

In [ ]:
# ============================================================
# [CELL 11] FILTER TO 15 TARGET STOCKS
# ============================================================
# SKIP IF: checkpoint 02_df_stocks already exists in GCS
#
# Reduce from 2.4M to ~129K rows by keeping only the 15
# selected stocks defined in [CELL 1].
#
# This reduction makes the dataset manageable for FinBERT
# inference (~1-2 hours for 118K articles on CPU).

if checkpoint_exists('02_df_stocks'):
    print('[SKIP] Checkpoint 02_df_stocks exists — loading from GCS...')
    ck        = load_checkpoint('02_df_stocks', spark=spark, load_spark=True)
    df_stocks = ck['spark_df']
    filtered_count = df_stocks.count()
else:
    df_stocks = df_clean.filter(F.col('Stock_symbol').isin(target_stocks))
    df_stocks.cache()
    filtered_count = df_stocks.count()
    row_count = df_clean.count()
    print(f'Filtered: {row_count:,} -> {filtered_count:,} rows')
    print(f'Kept: {filtered_count/row_count*100:.1f}% of full dataset')
    save_checkpoint('02_df_stocks', spark_df=df_stocks)

print(f'\nFiltered dataset shape: ({filtered_count:,} rows, {len(df_stocks.columns)} columns)')

print('\nArticle count per selected stock:')
df_stocks.groupBy('Stock_symbol') \
    .count() \
    .orderBy('count', ascending=False) \
    .show(20)

print('\nDate range and article count per stock:')
df_stocks.groupBy('Stock_symbol') \
    .agg(
        min('Date').alias('First_article'),
        max('Date').alias('Last_article'),
        F.count('*').alias('Total_articles')
    ) \
    .orderBy('Total_articles', ascending=False) \
    .show(20)

In [ ]:
# ============================================================
# [CELL 12] SAVE FILTERED DATASET TO GCS (PARQUET)
# ============================================================
# SKIP IF: Parquet file already exists in GCS
#
# Save as Parquet — standard format for Spark data storage.
# Advantages over CSV:
#   - Columnar format  -> fast reads for specific columns
#   - Compressed       -> ~3x smaller than equivalent CSV
#   - Schema preserved -> no re-parsing needed on reload
#   - Splittable       -> Spark reads partitions in parallel

if file_exists_in_gcs(PARQUET_PATH):
    print(f'[SKIP] Parquet already exists at {PARQUET_PATH}')
    df_verify = spark.read.parquet(PARQUET_PATH)
    print(f'       Verified: {df_verify.count():,} rows readable')
else:
    df_stocks.write \
        .option('header', 'true') \
        .mode('overwrite') \
        .parquet(PARQUET_PATH)
    print(f'Saved to: {PARQUET_PATH}')
    df_verify = spark.read.parquet(PARQUET_PATH)
    print(f'Verified: {df_verify.count():,} rows readable from Parquet')

print('\nSample rows from Parquet file:')
df_verify.show(5, truncate=50)

In [ ]:
# ============================================================
# [CELL 13] DOWNLOAD HISTORICAL PRICES (yfinance)
# ============================================================
# SKIP IF: checkpoint 03_df_joined already exists in GCS
#
# Download daily closing prices for all 15 stocks.
# Each news article needs today's closing price to compute
# whether an extreme price move (>1.5%) occurred NEXT day.
#
# PRICE DATA:
#   Source:   Yahoo Finance (yfinance library)
#   Period:   2008-2024 — matches FNSPID date range
#   Interval: 1d — one row per trading day per stock
#   Field:    Close price (adjusted for splits/dividends)

if checkpoint_exists('03_df_joined'):
    print('[SKIP] Checkpoint 03_df_joined exists — price download already done.')
    print('       Loading joined dataset from checkpoint...')
    ck        = load_checkpoint('03_df_joined', spark=spark, load_spark=True)
    df_joined = ck['spark_df']
    print(f'       Loaded: {df_joined.count():,} rows')
else:
    subprocess.run(['pip', 'install', 'yfinance', '--quiet'], check=True)
    import yfinance as yf

    fnspid_filt = spark.read.parquet(PARQUET_PATH)
    fnspid_filt.cache()
    print(f'Loaded filtered news dataset: {fnspid_filt.count():,} rows')

    print(f'\nDownloading prices for: {target_stocks}')
    prices = yf.download(
        tickers  = target_stocks,
        start    = '2008-01-01',
        end      = '2024-12-31',
        interval = '1d',
        progress = False
    )

    print(f'\nPrice data downloaded:')
    print(f'  Shape: {prices.shape}')
    print(f'  Date range: {prices.index.min().date()} -> {prices.index.max().date()}')
    print('\nFirst 5 rows (Close prices only):')
    print(prices['Close'].head())

In [ ]:
# ============================================================
# [CELL 14] VERIFY DATE RANGES & PREVIEW PRICES
# ============================================================
# SKIP IF: checkpoint 03_df_joined already exists in GCS
#
# Check that news date range and price date range overlap.
# Missing overlap = poor join = fewer matched rows.

if checkpoint_exists('03_df_joined'):
    print('[SKIP] Checkpoint 03_df_joined exists — date verification already done.')
else:
    print('Date range in news dataset:')
    fnspid_filt.select(min('Date'), max('Date')).show()

    print('Date range in price data:')
    print(f'  From: {prices.index.min().date()}')
    print(f'  To:   {prices.index.max().date()}')

    print('\nClose price statistics per stock:')
    print(prices['Close'].describe().round(2))

In [ ]:
# ============================================================
# [CELL 15] RESHAPE PRICES TO LONG FORMAT
# ============================================================
# SKIP IF: checkpoint 03_df_joined already exists in GCS
#
# yfinance returns prices in WIDE format:
#   index=Date, columns=[AAPL, GOOG, MSFT, ...]
# We need LONG format for the Spark join:
#   columns=[Date, Stock_symbol, Close_price]
#
# .melt() unpivots wide -> long:
#   id_vars    = Date (keep as row identifier)
#   var_name   = Stock_symbol (column headers become a column)
#   value_name = Close_price (values become a column)
#
# Example:
#   Wide: Date=2022-01-03 | AAPL=182.01 | GOOG=2893.59
#   Long: (2022-01-03, AAPL, 182.01)
#         (2022-01-03, GOOG, 2893.59)

if checkpoint_exists('03_df_joined'):
    print('[SKIP] Checkpoint 03_df_joined exists — reshape already done.')
else:
    prices_close = prices['Close'].reset_index()
    prices_long  = prices_close.melt(
        id_vars    = 'Date',
        var_name   = 'Stock_symbol',
        value_name = 'Close_price'
    )

    null_count  = prices_long['Close_price'].isna().sum()
    prices_long = prices_long.dropna(subset=['Close_price'])

    print(f'Reshaped prices: {prices_long.shape}')
    print(f'Missing prices dropped (weekends/holidays): {null_count:,}')
    print('\nSample rows (long format):')
    print(prices_long.head(10).to_string())

In [ ]:
# ============================================================
# [CELL 16] JOIN NEWS WITH PRICES (SPARK)
# ============================================================
# SKIP IF: checkpoint 03_df_joined already exists in GCS
#
# Join news articles with daily stock prices on:
#   (Date, Stock_symbol)
#
# JOIN TYPE: inner join
#   -> keeps only articles from market-open trading days
#   -> drops ~9% of articles (weekends + market holidays)
#   -> expected — no price available = no prediction possible
#
# DATE NORMALIZATION:
#   News dates:  '2022-06-03 14:32:00' (TimestampType)
#   Price dates: '2022-06-03' (DateType)
#   Both normalized to DateType so join keys match exactly.

if checkpoint_exists('03_df_joined'):
    print('[SKIP] Checkpoint 03_df_joined exists — join already done.')
    print(f'       df_joined already loaded: {df_joined.count():,} rows')
else:
    prices_spark = spark.createDataFrame(prices_long)
    fnspid_filt  = fnspid_filt.withColumn('Date',  to_date('Date'))
    prices_spark = prices_spark.withColumn('Date', to_date('Date'))

    df_joined    = fnspid_filt.join(prices_spark, on=['Date', 'Stock_symbol'], how='inner')
    joined_count = df_joined.count()
    news_count   = fnspid_filt.count()

    print(f'Before join: {news_count:,} articles')
    print(f'After join:  {joined_count:,} articles')
    print(f'Match rate:  {joined_count/news_count*100:.1f}%')
    print(f'Lost rows:   {news_count-joined_count:,} (weekend/holiday articles)')

    save_checkpoint('03_df_joined', spark_df=df_joined)

In [ ]:
# ============================================================
# [CELL 17] INSPECT JOINED DATASET
# ============================================================
# Always runs — verify the join result regardless of checkpoint.
# Check: schema, sample rows, price statistics, stock counts.

print('Joined dataset schema:')
df_joined.printSchema()

print(f'\nJoined dataset shape: ({df_joined.count():,} rows, {len(df_joined.columns)} columns)')

print('\nSample rows (vertical for readability):')
df_joined.show(5, truncate=60, vertical=True)

print('\nClose price statistics:')
df_joined.select(
    F.min('Close_price').alias('Min_price'),
    F.max('Close_price').alias('Max_price'),
    F.avg('Close_price').alias('Avg_price'),
    F.stddev('Close_price').alias('Std_price')
).show()

print('\nArticles per stock after join:')
df_joined.groupBy('Stock_symbol') \
    .agg(
        F.count('*').alias('Articles'),
        F.avg('Close_price').alias('Avg_close')
    ) \
    .orderBy('Articles', ascending=False) \
    .show(20)

print('\nNull values per column:')
df_joined.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_joined.columns]).show()

In [ ]:
# ============================================================
# [CELL 18] CONVERT JOINED DATASET TO PANDAS
# ============================================================
# SKIP IF: checkpoint 04_pd_news already exists in GCS
#
# Collect the Spark DataFrame to the driver node as pandas.
# Required for FinBERT inference — transformer models cannot
# be easily distributed without a GPU cluster.
#
# WHY IS THIS SAFE?
#   118K rows x ~8 columns ~ 500MB in memory.
#   The Dataproc driver node has sufficient RAM.
#   We would NOT do this on the full 2.4M row dataset.

if file_exists_in_gcs(f'{CHECKPOINT_DIR}/04_pd_news.csv'):
    print('[SKIP] Checkpoint 04_pd_news exists — loading FinBERT results from GCS...')
    ck      = load_checkpoint('04_pd_news', load_pandas=True)
    pd_news = ck['pandas_df']
    print(f'\nLoaded labeled dataset: {pd_news.shape[0]:,} rows x {pd_news.shape[1]} columns')
    print(f'Columns: {pd_news.columns.tolist()}')
    print('\nSentiment distribution:')
    counts = pd_news['sentiment'].value_counts()
    pcts   = pd_news['sentiment'].value_counts(normalize=True).mul(100).round(1)
    for label in counts.index:
        bar = '|' * int(pcts[label] / 2)
        print(f'  {label:<10} {counts[label]:>7,} ({pcts[label]:>5.1f}%)  {bar}')
else:
    pd_news = df_joined.toPandas()
    print(f'Converted to pandas: {pd_news.shape[0]:,} rows x {pd_news.shape[1]} columns')
    print(f'Columns: {pd_news.columns.tolist()}')
    print(f'\nMemory usage: {pd_news.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
    print('\nNull values per column:')
    print(pd_news.isnull().sum())
    print('\nFirst 3 rows:')
    print(pd_news.head(3).to_string())

In [ ]:
# ============================================================
# [CELL 19] VADER SENTIMENT (BASELINE COMPARISON)
# ============================================================
# SKIP IF: checkpoint 04_pd_news already exists
#          (VADER was already run and saved)
#
# WHAT IS VADER?
#   VADER (Valence Aware Dictionary and sEntiment Reasoner)
#   is a rule-based analyzer using a dictionary of ~7,500
#   scored words. Designed for social media text.
#   Compound score in [-1, +1]:
#     >= +0.05 -> Positive
#     <= -0.05 -> Negative
#     Between  -> Neutral
#
# LIMITATION FOR FINANCIAL TEXT:
#   VADER does not understand financial jargon:
#     'beat earnings estimates' -> VADER: neutral/negative (?)
#                                  FinBERT: POSITIVE (correct)
#     'raised guidance'         -> VADER: likely neutral
#                                  FinBERT: POSITIVE (correct)
#     'missed revenue targets'  -> VADER: partial recognition
#                                  FinBERT: NEGATIVE (correct)
#
# We compare both models to demonstrate FinBERT's superiority.

if file_exists_in_gcs(f'{CHECKPOINT_DIR}/04_pd_news.csv'):
    print('[SKIP] Checkpoint 04_pd_news exists — VADER already ran.')
    if 'vader_sentiment' in pd_news.columns:
        print('\nVADER sentiment distribution (from checkpoint):')
        counts = pd_news['vader_sentiment'].value_counts()
        pcts   = pd_news['vader_sentiment'].value_counts(normalize=True).mul(100).round(1)
        for label in counts.index:
            bar = '|' * int(pcts[label] / 2)
            print(f'  {label:<10} {counts[label]:>7,} ({pcts[label]:>5.1f}%)  {bar}')
    else:
        print('(vader_sentiment column not found in checkpoint)')
else:
    subprocess.run(['pip', 'install', 'vaderSentiment', '--quiet'], check=True)
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

    analyzer = SentimentIntensityAnalyzer()

    pd_news['vader_score'] = pd_news['Lsa_summary'].apply(
        lambda x: analyzer.polarity_scores(str(x))['compound'] if pd.notna(x) else 0
    )

    def classify_vader(score):
        if score >= 0.05:    return 'positive'
        elif score <= -0.05: return 'negative'
        else:                return 'neutral'

    pd_news['vader_sentiment'] = pd_news['vader_score'].apply(classify_vader)

    print('VADER sentiment distribution:')
    counts = pd_news['vader_sentiment'].value_counts()
    pcts   = pd_news['vader_sentiment'].value_counts(normalize=True).mul(100).round(1)
    for label in counts.index:
        bar = '|' * int(pcts[label] / 2)
        print(f'  {label:<10} {counts[label]:>7,} ({pcts[label]:>5.1f}%)  {bar}')

    print('\nSample VADER-labeled articles:')
    print(pd_news[['Lsa_summary', 'vader_score', 'vader_sentiment']].head(10).to_string())

In [ ]:
# ============================================================
# [CELL 20] INSPECT VADER RESULTS MANUALLY
# ============================================================
# SKIP IF: checkpoint 04_pd_news already exists
#
# Manually inspect positive and negative VADER examples.
# Look for cases where VADER misses financial context —
# these become evidence for preferring FinBERT.

if file_exists_in_gcs(f'{CHECKPOINT_DIR}/04_pd_news.csv'):
    print('[SKIP] Checkpoint 04_pd_news exists — VADER inspection already done.')
else:
    print('=' * 60)
    print('VADER NEGATIVE examples:')
    print('=' * 60)
    for text in pd_news[pd_news['vader_sentiment'] == 'negative']['Lsa_summary'].head(3):
        print(f'\n  {str(text)[:300]}')
        print()

    print('=' * 60)
    print('VADER POSITIVE examples:')
    print('=' * 60)
    for text in pd_news[pd_news['vader_sentiment'] == 'positive']['Lsa_summary'].head(3):
        print(f'\n  {str(text)[:300]}')
        print()

    print('VADER score statistics:')
    print(pd_news['vader_score'].describe().round(3))

In [ ]:
# ============================================================
# [CELL 21] FINBERT SENTIMENT (FINAL MODEL)
# ============================================================
# SKIP IF: checkpoint 04_pd_news already exists in GCS
#
# WHY FINBERT OVER VADER?
#   FinBERT is a BERT transformer fine-tuned on ~10K financial
#   sentences from the Financial PhraseBank dataset.
#   It outperforms VADER because:
#     1. Domain-specific training on financial news text
#     2. Context-aware — understands full sentence meaning
#     3. Handles negation, hedging, and financial jargon
#     4. Higher correlation with actual price returns
#
# INPUT:  Lsa_summary[:512] — truncated to BERT token limit
# OUTPUT: label (positive/neutral/negative) + confidence score
#
# RUNTIME: ~1-2 hours on CPU for 118K articles.
# PROGRESS: saved every 1,000 articles — resumes if interrupted.

if file_exists_in_gcs(f'{CHECKPOINT_DIR}/04_pd_news.csv'):
    print('[SKIP] Checkpoint 04_pd_news exists — FinBERT already ran.')
    print('\nFinBERT sentiment distribution (from checkpoint):')
    counts = pd_news['sentiment'].value_counts()
    pcts   = pd_news['sentiment'].value_counts(normalize=True).mul(100).round(1)
    for label in counts.index:
        bar = '|' * int(pcts[label] / 2)
        print(f'  {label:<10} {counts[label]:>7,} ({pcts[label]:>5.1f}%)  {bar}')

    print('\nAverage confidence score by sentiment:')
    print(pd_news.groupby('sentiment')['confidence'].describe().round(3))

    print('\nSentiment distribution per stock:')
    print(pd_news.groupby(['Stock_symbol', 'sentiment'])
          .size()
          .unstack(fill_value=0))

    if 'vader_sentiment' in pd_news.columns:
        agreement = (pd_news['vader_sentiment'] == pd_news['sentiment']).mean()
        print(f'\nVADER vs FinBERT agreement rate: {agreement*100:.1f}%')
        print(f'Disagreement: {(1-agreement)*100:.1f}% — cases where financial context matters')
else:
    subprocess.run(['pip', 'install', 'transformers', 'torch', '--quiet'])
    from transformers import pipeline
    from tqdm import tqdm
    tqdm.pandas()

    print('Loading FinBERT (ProsusAI/finbert)...')
    finbert = pipeline(
        'text-classification',
        model     = 'ProsusAI/finbert',
        tokenizer = 'ProsusAI/finbert'
    )
    print('FinBERT loaded!')

    print('\nSample inputs to FinBERT:')
    for i, text in enumerate(pd_news['Lsa_summary'].dropna().head(3)):
        print(f'\n  [{i+1}] {str(text)[:300]}...')

    def get_sentiment(text):
        """
        Run FinBERT on a single text string.
        Returns (label, confidence_score).
          label:      positive | neutral | negative
          confidence: model certainty in [0, 1]
        text[:512] truncates to stay within BERT token limit.
        """
        if not text or str(text) == 'NULL' or pd.isna(text):
            return None, None
        result = finbert(str(text)[:512])[0]
        return result['label'], result['score']

    # ── Progress-saving inference loop ───────────────────────
    PROGRESS_CSV = '/tmp/pd_news_finbert_progress.csv'
    GCS_PROGRESS = f'{CHECKPOINT_DIR}/04_pd_news_progress.csv'
    SAVE_EVERY   = 1000

    if os.path.exists(PROGRESS_CSV):
        pd_done      = pd.read_csv(PROGRESS_CSV, index_col=0)
        done_indices = set(pd_done.index)
        print(f'\nResuming: {len(pd_done):,} articles already processed')
    else:
        pd_done      = pd.DataFrame()
        done_indices = set()
        print(f'\nStarting fresh: {len(pd_news):,} articles to process')

    pd_pending = pd_news[~pd_news.index.isin(done_indices)].copy()
    print(f'Articles remaining: {len(pd_pending):,}')

    results = []
    for i, (idx, row) in enumerate(tqdm(pd_pending.iterrows(), total=len(pd_pending))):
        label, score = get_sentiment(row['Lsa_summary'])
        results.append({'index': idx, 'sentiment': label, 'confidence': score})

        if (i + 1) % SAVE_EVERY == 0:
            df_partial = pd.DataFrame(results).set_index('index')
            pd_news.loc[df_partial.index, ['sentiment', 'confidence']] = \
                df_partial[['sentiment', 'confidence']]
            pd_news.to_csv(PROGRESS_CSV, index=True)
            os.system(f'gcloud storage cp {PROGRESS_CSV} {GCS_PROGRESS}')
            print(f'  Checkpoint: {i+1:,}/{len(pd_pending):,} saved')

    df_final = pd.DataFrame(results).set_index('index')
    pd_news.loc[df_final.index, ['sentiment', 'confidence']] = \
        df_final[['sentiment', 'confidence']]

    pd_news = pd_news.dropna(subset=['sentiment', 'Close_price'])

    print(f'\nFinBERT complete! Final dataset: {pd_news.shape[0]:,} rows')

    print('\nFinBERT sentiment distribution:')
    counts = pd_news['sentiment'].value_counts()
    pcts   = pd_news['sentiment'].value_counts(normalize=True).mul(100).round(1)
    for label in counts.index:
        bar = '|' * int(pcts[label] / 2)
        print(f'  {label:<10} {counts[label]:>7,} ({pcts[label]:>5.1f}%)  {bar}')

    print('\nAverage confidence score by sentiment:')
    print(pd_news.groupby('sentiment')['confidence'].describe().round(3))

    if 'vader_sentiment' in pd_news.columns:
        agreement = (pd_news['vader_sentiment'] == pd_news['sentiment']).mean()
        print(f'\nVADER vs FinBERT agreement rate: {agreement*100:.1f}%')
        print(f'Disagreement: {(1-agreement)*100:.1f}% — cases where financial context matters')

    print('\nSentiment distribution per stock:')
    print(pd_news.groupby(['Stock_symbol', 'sentiment'])
          .size()
          .unstack(fill_value=0))

    print('\nSample FinBERT-labeled articles:')
    print(pd_news[['Stock_symbol', 'Date', 'sentiment', 'confidence', 'Lsa_summary']]
          .head(5).to_string(max_colwidth=100))

In [ ]:
# ============================================================
# [CELL 22] SAVE FINAL CHECKPOINT TO GCS
# ============================================================
# Save the complete FinBERT-labeled dataset to GCS.
# This is the entry point for Part 2 (ML pipeline).
#
# Loading this checkpoint skips:
#   - All Spark preprocessing:   ~10 min on 2.4M rows
#   - FinBERT inference:         ~1-2 hours on 118K rows
# Total time saved on subsequent runs: ~2 hours.
#
# To load in Part 2 (ML pipeline):
#   os.system('gcloud storage cp gs://assesment2-dc/checkpoints/04_pd_news.csv /tmp/pd_news.csv')
#   pd_news = pd.read_csv('/tmp/pd_news.csv')

if file_exists_in_gcs(f'{CHECKPOINT_DIR}/04_pd_news.csv'):
    print('[SKIP] Checkpoint 04_pd_news already exists in GCS.')
    print('       No need to save again — dataset unchanged.')
else:
    save_checkpoint('04_pd_news', pandas_df=pd_news)

# ── Final pipeline summary (always shown) ────────────────────
print('\n' + '=' * 60)
print('PIPELINE COMPLETE — SUMMARY')
print('=' * 60)
print(f'  Raw dataset:              ~2,467,690 articles')
print(f'  After 15-stock filter:    ~129,000 articles')
print(f'  After price join (91%):   ~118,400 articles')
print(f'  After FinBERT labeling:    {pd_news.shape[0]:,} articles')
print(f'  Columns:                   {pd_news.shape[1]}')
print(f'\n  Sentiment breakdown:')
for label, count in pd_news['sentiment'].value_counts().items():
    pct = count / pd_news.shape[0] * 100
    print(f'    {label:<10} {count:>7,}  ({pct:.1f}%)')
print(f'\n  Checkpoint: {CHECKPOINT_DIR}/04_pd_news.csv')
print(f'\n  To load in Part 2 (ML Pipeline):')
print(f'    os.system("gcloud storage cp {CHECKPOINT_DIR}/04_pd_news.csv /tmp/pd_news.csv")')
print(f'    pd_news = pd.read_csv("/tmp/pd_news.csv")')
print('=' * 60)